# 03 — Backtest basics

Trace one native backtest from data and forecasts to positions, costs, and a full-universe carry/EWMAC portfolio.

## Anatomy of a native backtest

pysystemtrade backtests are a chain of *stages*, each cacheable and
inspectable. This notebook builds a small four-instrument system and pulls
every stage apart so you know exactly where any number comes from.

```
rawdata -> rules -> forecastScaleCap -> combForecast -> positionSize -> portfolio -> accounts
```

We start from the book's "chapter 15" configuration (6 EWMAC speeds +
carry, fixed forecast weights) and override just the universe and capital.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from systems.provided.futures_chapter15.basesystem import futures_system

INSTRUMENTS = ["SHFE_RB", "CFFEX_IF", "DCE_M", "CZCE_TA"]
PLOT_START = "2015-01-01"

config = Config("systems.provided.futures_chapter15.futuresconfig.yaml")
config.instruments = INSTRUMENTS
config.instrument_weights = {code: 1 / len(INSTRUMENTS) for code in INSTRUMENTS}
config.notional_trading_capital = 10_000_000
config.base_currency = "CNH"          # everything in renminbi; FX == 1

system = futures_system(data=dbFuturesSimData(), config=config)
print(system)

## Stage 1: `rawdata` — prices and volatility

Everything downstream is *risk-scaled*, so the first thing the system
computes is a robust daily return volatility (EWMA with a floor). The adjusted
price is an additive-Panama level: price differences are meaningful, but its
absolute vertical level is arbitrary.

In [ ]:
code_ = "SHFE_RB"
prices = system.rawdata.get_daily_prices(code_)
vol = system.rawdata.daily_returns_volatility(code_)
plot_end = prices.index[-1]
plot_label = f"{PLOT_START} to {plot_end:%Y-%m-%d}"

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
prices.loc[PLOT_START:].plot(
    ax=axes[0], title=f"{code_} additive-Panama level, {plot_label}")
axes[0].set_ylabel("adjusted price units")
vol.loc[PLOT_START:].plot(
    ax=axes[1], title="daily return volatility")
axes[1].set_ylabel("price units per day")
axes[1].set_xlabel("date")
plt.tight_layout()

## Stage 2: `rules` — raw forecasts

A trading rule is just a function of data → signal. EWMAC divides a moving
average crossover by volatility, so forecasts are comparable across
instruments; carry annualises the price gap between the carry contract and
the priced contract.

In [ ]:
ewmac_raw = system.rules.get_raw_forecast(code_, "ewmac64_256")
carry_raw = system.rules.get_raw_forecast(code_, "carry")
raw_forecasts = pd.concat(
    {"ewmac64_256": ewmac_raw, "carry": carry_raw}, axis=1
).loc[PLOT_START:]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
raw_forecasts["ewmac64_256"].plot(
    ax=axes[0], title=f"raw EWMAC forecast, {code_}, {plot_label}")
axes[0].set_ylabel("raw EWMAC units")
raw_forecasts["carry"].plot(
    ax=axes[1], title=f"raw carry forecast, {code_}")
axes[1].set_ylabel("raw carry units")
axes[1].set_xlabel("date")
plt.tight_layout()

## Stage 3: `forecastScaleCap` — scale to ±10, cap at ±20

Raw forecasts have arbitrary units. A *forecast scalar* is an externally
calibrated number intended to put a rule near an average absolute forecast of
10 over a broad reference population; it is not fitted to this instrument or
guaranteed to hit exactly 10 in this sample. Forecasts are then capped at ±20.

In [ ]:
scaled = system.forecastScaleCap.get_capped_forecast(code_, "carry")
print(f"carry forecast scalar (fixed in config): "
      f"{float(system.forecastScaleCap.get_forecast_scalar(code_, 'carry').iloc[-1]):.1f}")
print(f"observed average |scaled forecast| for {code_}: "
      f"{scaled.dropna().abs().mean():.2f}  (external calibration target: 10)")
ax = scaled.loc[PLOT_START:].plot(
    title=f"scaled and capped carry forecast, {code_}, {plot_label}")
ax.set_ylabel("forecast units")
ax.set_xlabel("date")

## Stage 4: `combForecast` — one forecast per instrument

Rules are combined with *forecast weights*, then multiplied by an externally
calibrated *forecast diversification multiplier* (FDM). Its population target
is again an average magnitude near 10, not an exact in-sample identity.
Chapter 15 uses fixed weights — note the fast EWMACs get zero weight (they
trade too much for their edge; costs).

In [ ]:
print("forecast weights (fixed):", system.config.forecast_weights)
print("externally calibrated FDM:", system.config.forecast_div_multiplier)
combined = system.combForecast.get_combined_forecast(code_)
print(f"observed average |combined forecast| for {code_}: "
      f"{combined.dropna().abs().mean():.2f}")
ax = combined.loc[PLOT_START:].plot(
    title=f"combined forecast, {code_}, {plot_label}")
ax.set_ylabel("forecast units")
ax.set_xlabel("date")

## Stage 5: `positionSize` — from forecast to contracts

The volatility-targeting engine: capital × risk target defines a daily cash
volatility budget; an instrument's *block value* (point size × price × vol)
says how much risk one contract carries; the ratio is the position for an
average-strength (=10) forecast, scaled by the actual forecast.

In [ ]:
subsystem_position = system.positionSize.get_subsystem_position(code_)
print(f"capital: {system.positionSize.get_notional_trading_capital():,.0f} CNH, "
      f"risk target {system.positionSize.get_percentage_vol_target():.0f}%/yr")
ax = subsystem_position.loc[PLOT_START:].plot(
    title=f"subsystem position, {code_}, {plot_label}")
ax.set_ylabel("contracts")
ax.set_xlabel("date")

## Stages 6-7: `portfolio` and `accounts`

Instrument weights (fixed 25% each here) and the instrument diversification
multiplier scale subsystem positions into portfolio positions; `accounts`
computes P&L net of costs (Chinese spread costs come from the imported
half-spreads; commissions are ~zero and modelled as zero). The continuous
target is not what the backtest necessarily holds: buffering avoids small
trades and the account stage rounds the result to whole contracts.

In [ ]:
target_position = system.portfolio.get_notional_position(code_)
buffered_position = system.accounts.get_buffered_position(code_)
instrument_curve = system.accounts.pandl_for_instrument(code_)
held_position = instrument_curve.pandl_calculator_with_costs.positions
position_comparison = pd.concat(
    {
        "continuous target": target_position,
        "buffered desired (pre-fill)": buffered_position,
        "held after delayed fill": held_position,
    },
    axis=1,
).loc[PLOT_START:]
ax = position_comparison.plot(
    title=f"portfolio target versus buffered position, {code_}, {plot_label}")
ax.set_ylabel("contracts")
ax.set_xlabel("date")

In [ ]:
portfolio_pandl = system.accounts.portfolio()
plotted_portfolio = R.rewrap(portfolio_pandl.percent.as_ts.loc[PLOT_START:])
portfolio_curve = R.cumulative_from_zero(plotted_portfolio.percent.as_ts)
ax = portfolio_curve.plot(
    title=f"four-instrument portfolio cumulative return, {plot_label}")
ax.set_ylabel("percent of capital")
ax.set_xlabel("date")
stats = dict(plotted_portfolio.stats()[0])
{key: stats[key] for key in ["ann_mean", "ann_std", "sharpe", "sortino",
                             "avg_drawdown", "skew", "t_stat", "p_value"]}

## The account-curve toolkit

`accounts` methods return `accountCurve` / `accountCurveGroup` objects —
pandas Series with extra powers, chainable:

- `.gross` / `.net` / `.costs` — cost decomposition
- `.percent` / `.value_terms` — % of capital vs currency
- `.daily` / `.weekly` / `.monthly` / `.annual` — resampling
- `.stats()`, `.sharpe()`, `.t_stat()`, `.p_value()`, `curve()` ...

In [ ]:
one_instrument = system.accounts.pandl_for_instrument("SHFE_RB")
daily_decomposition = pd.DataFrame({
    "gross": one_instrument.percent.gross.as_ts,
    "net": one_instrument.percent.net.as_ts,
    "cumulative costs": one_instrument.percent.costs.as_ts,
}).loc[PLOT_START:]
cost_decomposition = R.cumulative_from_zero(daily_decomposition)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
cost_decomposition[["gross", "net"]].plot(
    ax=axes[0], title=f"SHFE_RB gross and net cumulative P&L, {plot_label}")
axes[0].set_ylabel("percent of capital")
cost_decomposition["cumulative costs"].plot(
    ax=axes[1], color="tab:red", title="cumulative trading costs")
axes[1].set_ylabel("percent of capital")
axes[1].set_xlabel("date")
plt.tight_layout()

## Two gotchas worth learning early

1. `portfolio()` returns an `accountCurveGroup`; its `[...]` indexes by
   **instrument name** (`group["SHFE_RB"]`), never by date.
2. An individual `accountCurve` subclasses `pd.Series`, and pandas date
   slicing returns a **plain Series** — the stats methods silently vanish.
   The supported way back is `account_curve_from_returns` (wrapped as
   `R.rewrap`). You'll use this constantly for subperiod analysis.

In [ ]:
rebar_curve = portfolio_pandl["SHFE_RB"]      # group indexing: by instrument
print(f"group['SHFE_RB'] is a {type(rebar_curve).__name__}, "
      f"sharpe {rebar_curve.sharpe():.3f}")

sliced = rebar_curve["2020-01-01":]           # date slicing degrades it
print("date-sliced type:", type(sliced).__name__)
try:
    sliced.sharpe()
except AttributeError as error:
    print("AttributeError:", error)

recent = R.rewrap(rebar_curve.percent.as_ts["2020-01-01":])
print(f"SHFE_RB 2020+ sharpe, properly rewrapped: {recent.sharpe():.2f}")

**Next**: the full-universe section below scales up the same native stages.

## The full Chinese universe

Time to run trend and carry across every stored history. Two things matter
before the result: market membership must use only information available at
the time, and a many-instrument account curve must be decomposed as actual
capital contributions.

## Config exclusions are not a historical universe

Two mechanisms, for two intents:

1. **Per-study exclusions** — pass a list to `config.instruments`. This is
   appropriate for a declared experiment, but a list chosen today creates
   survivor bias if projected over the whole backtest.
2. **Standing config exclusions** — `config.exclude_instrument_lists`:
   - `ignore_instruments`: the system *cannot see them at all* (removed
     inside `get_instrument_list()`);
   - `bad_markets` / `trading_restrictions`: flags used by production and
     dynamic optimisation. In a static-weight backtest they do **not** by
     themselves turn an existing weight into zero. Add the instrument to
     `allocate_zero_instrument_weights_to_these_instruments`, or set its
     explicit weight to zero.

The cell below demonstrates the standing mechanism without running any P&L
(instrument lists are cheap; nothing else is computed).

In [ ]:
from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from systems.provided.futures_chapter15.basesystem import futures_system

demo_universe = R.chinese_universe(dbFuturesSimData())
demo_config = Config("systems.provided.futures_chapter15.futuresconfig.yaml")
demo_config.instruments = demo_universe
# TRAP: the chapter-15 yaml carries instrument_weights for its own six
# instruments, and instrument_weights BEATS instruments when the system
# picks its universe. Override it or the requested universe silently becomes 6.
demo_config.instrument_weights = {code: 1 / len(demo_universe)
                                  for code in demo_universe}
demo_config.exclude_instrument_lists = dict(
    ignore_instruments=["CZCE_JR", "CZCE_LR"],   # pretend we never heard of them
    trading_restrictions=[],
    bad_markets=["SHFE_WR"],                     # flag it; no automatic zero here
)
demo_config.allocate_zero_instrument_weights_to_these_instruments = ["SHFE_WR"]
demo_system = futures_system(data=dbFuturesSimData(), config=demo_config)
print(f"requested: {len(demo_universe)}")
print(f"visible to the system (ignore list removed): "
      f"{len(demo_system.get_instrument_list())}")
print(f"flagged bad (still simulated): "
      f"{demo_system.get_list_of_bad_markets()}")

## A lagged liquidity rule, not a 2026 survivor list

`R.chinese_universe(data)` returns all 95 stitched histories, including old
names and markets whose liquidity later died. We reconstruct the held
contract's reported volume and require seasoning plus a rolling volume
threshold using observations through each close. The result is a Boolean
eligibility panel: an old market may receive capital while it was genuinely
liquid and then leave; a new market joins only after it has enough observable
history. Native `delayfill=True` shifts a target by one system business row,
so today's close and volume can never change the return just observed. On an
exchange holiday that business row has no fresh quote; the upstream account
then approximates execution with a stale or missing price before the next
observed session. We disclose that limitation rather than adding another
account engine.

The declared research rule is intentionally plain: a 20-observed-session
mean enters at 130 contracts and exits below 70, providing hysteresis; known
terminal histories are targeted flat before their final stored close so the
native account can book the closing trade and cost. These are capacity-proxy
assumptions, not optimised parameters.

This is still a research proxy. Daily volume is not order-book depth, open
interest, exchange limits, or proof that a 100m-CNH account could fill. The
event table makes every entry and exit auditable rather than hiding them in a
hard-coded "dead market" list.

## The native system: six EWMAC speeds + carry

Chapter-15 configuration again: fixed forecast weights (the two fastest
EWMACs get zero — too expensive), carry at 50%. Equal instrument weights
for now; estimation is notebook 04's subject. `base_currency: CNH`, 100m
CNH notional. The native system still constructs forecasts, volatility,
positions and modelled costs. The small research helper only gates and
renormalises native portfolio weights through time before positions are sized.

One deliberate non-default is `vol_normalise_currency_costs=False`. The
standard option rescales all historical cash costs using each instrument's
final 180-day price volatility; that is future-informed and becomes zero or
undefined for some terminal zombie histories. Here configured commissions and
spreads are charged directly at native fills, without that ex-post rescaling;
notebooks 04–06 keep the same research choice for comparable PIT portfolios.

In [ ]:
from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from systems.accounts.accounts_stage import Account
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecasting import Rules
from systems.positionsizing import PositionSizing
from systems.rawdata import RawData

data = dbFuturesSimData()
# Keep every stored Chinese history.  A lagged, point-in-time liquidity mask
# decides when each market can receive capital; today's survivor list is never
# projected backwards.
universe = R.chinese_universe(data)
held_volume = R.held_contract_volumes(data, universe)
liquidity = R.liquidity_eligibility(held_volume, force_terminal_close=True)
liquidity_events = R.liquidity_event_table(liquidity, held_volume)

config = Config("systems.provided.futures_chapter15.futuresconfig.yaml")
config.instruments = universe
config.instrument_weights = {code: 1 / len(universe) for code in universe}
config.instrument_div_multiplier = 2.5   # IDM at its cap; estimation is in notebook 04
config.notional_trading_capital = 100_000_000
config.base_currency = "CNH"
# Research choice for ended histories: charge configured cash/spread costs as
# observed, without the default ex-post final-volatility cost normalisation.
config.vol_normalise_currency_costs = False

system = System(
    [
        Account(),
        R.PointInTimePortfolios(liquidity),
        PositionSizing(),
        RawData(),
        ForecastCombine(),
        ForecastScaleCap(),
        Rules(),
    ],
    data,
    config,
)
print(f"{len(universe)} instruments, rules: {list(config.trading_rules.keys())}")

In [ ]:
active_count = liquidity.sum(axis=1)
average_volume = R.trailing_liquidity(held_volume)
latest_average_volume = average_volume.reindex(liquidity.index).ffill().iloc[-1]
latest_active_volume = latest_average_volume[liquidity.iloc[-1]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
active_count.plot(ax=axes[0], title="point-in-time eligible markets")
axes[0].set_ylabel("market count")
np.log1p(latest_active_volume).plot(
    ax=axes[1], kind="hist", bins=30,
    title="latest active markets: held-contract volume")
axes[1].set_xlabel("log(1 + 20-session mean contracts); zeros stay at zero")
plt.tight_layout()

print(f"eligibility dates: {liquidity.index[0].date()} to "
      f"{liquidity.index[-1].date()}; latest active: {int(active_count.iloc[-1])}")
display(liquidity_events.tail(12))

In [ ]:
portfolio_pandl = system.accounts.portfolio()
curve = portfolio_pandl.percent
R.cumulative_from_zero(curve.as_ts).plot(
    title=f"EWMAC+carry, {len(system.get_instrument_list())} "
          "stored Chinese histories; point-in-time eligible")
R.stats_table({"point-in-time equal weight, net": portfolio_pandl})

## Where does it come from? Asset-class decomposition

The safest additive decomposition starts from cash P&L. Divide every
instrument's `value_terms` by the **same whole-portfolio** 100m-CNH notional,
then sum the resulting percentage-point contributions. This avoids relying on
the account curve's percentage-view conventions and gives us a direct daily
reconciliation to the portfolio.

Zeros are real observations: a closed weekday has zero P&L. We use
`sum(min_count=1)` so an all-missing row remains missing without converting
genuine zero-return days to `NaN`.

In [ ]:
notional = system.accounts.get_notional_capital()
instrument_contributions = (
    portfolio_pandl.value_terms.to_frame() / notional * 100
)
portfolio_return = portfolio_pandl.value_terms.as_ts / notional * 100
reconciled = instrument_contributions.sum(axis=1, min_count=1)
comparison = pd.concat([portfolio_return.rename("portfolio"),
                        reconciled.rename("sum of instruments")], axis=1).dropna()
assert (comparison["portfolio"] - comparison["sum of instruments"]).abs().max() < 1e-8

asset_class = dict(data.get_instrument_asset_classes())
by_class = instrument_contributions.T.groupby(
    pd.Series(asset_class)
).sum(min_count=1).T

R.cumulative_from_zero(by_class).plot(
    title="cumulative contribution by asset class (% of capital)")
R.stats_table({name: R.rewrap(series.dropna())
               for name, series in by_class.items()})
print(f"maximum daily reconciliation error: "
      f"{(comparison['portfolio'] - comparison['sum of instruments']).abs().max():.3g}")

## Per-instrument contribution Sharpe: describe the cross-section

With 95 histories you should think in distributions, not favourites. Markets
within one portfolio share rules, macro shocks and often contracts in the
same commodity complex, so a one-sample t-test that pretends 95 independent
draws is not valid inference. We report the median, interquartile range and
asset-class medians. These describe this backtest; they do not estimate a
population p-value.

In [ ]:
import warnings

# Each series is its actual contribution to whole-portfolio capital. A market
# that was never eligible has zero variance and therefore undefined Sharpe.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="invalid value encountered in scalar divide",
        category=RuntimeWarning,
    )
    sharpes = pd.Series({
        code: R.rewrap(series.dropna()).sharpe()
        for code, series in instrument_contributions.items()
    })

ordered_sharpes = sharpes.sort_values()
ax = ordered_sharpes.plot.barh(
    figsize=(9, 18),
    color=np.where(ordered_sharpes < 0, "firebrick", "steelblue"),
)
ax.set_title("net Sharpe of each instrument's actual portfolio contribution")
ax.set_xlabel("Sharpe ratio")
ax.set_ylabel("instrument")

clean = sharpes.dropna()
summary = pd.Series({
    "defined instruments": len(clean),
    "median": clean.median(),
    "25th percentile": clean.quantile(0.25),
    "75th percentile": clean.quantile(0.75),
    "negative": int((clean < 0).sum()),
})
display(summary.to_frame("contribution Sharpe"))

class_summary = pd.DataFrame({"sharpe": clean}).assign(
    asset_class=pd.Series(asset_class)
).dropna().groupby("asset_class")["sharpe"].agg(
    count="count", median="median",
    q25=lambda values: values.quantile(0.25),
    q75=lambda values: values.quantile(0.75),
)
class_summary

## Costs sanity check

The standardised "Sharpe-ratio cost per trade" makes instruments comparable;
Rob's rule of thumb is to investigate markets above roughly 0.01. This is a
latest-data diagnostic, not a historical liquidity screen. The portfolio
curve above uses each contract's configured cash/spread cost on its actual
trades.

In [ ]:
sr_costs = pd.Series({
    code: system.accounts.get_SR_cost_per_trade_for_instrument(code)
    for code in system.get_instrument_list()
}).sort_values()
latest_active = liquidity.iloc[-1]
active_costs = sr_costs[latest_active.reindex(sr_costs.index).fillna(False)]
cheapest = active_costs.head(5)
most_expensive = active_costs.tail(5).sort_values(ascending=False)
ranked_costs = pd.DataFrame({
    "cheapest instrument": cheapest.index,
    "cheapest SR cost": cheapest.to_numpy(),
    "most expensive instrument": most_expensive.index,
    "most expensive SR cost": most_expensive.to_numpy(),
}, index=pd.RangeIndex(1, 6, name="rank"))
display(ranked_costs)
print(f"finite SR costs: {np.isfinite(sr_costs).sum()} of {len(sr_costs)} stored; "
      f"latest-active above 0.01: {(active_costs > 0.01).sum()} "
      f"of {len(active_costs)}")

Read the figures produced by this run, not a result sentence copied from an
older data snapshot. Breadth and cost diagnostics can support the design;
they do not turn the best individual backtests into a selection rule. The
liquidity thresholds are also assumptions to stress, not fitted truth.

**Next**: notebook 04 separates weighting from pooling; notebook 05 then asks
whether carry or trend did the work.